# finchGE starter notebook

###  String Matching with Grammatical Evolution on Jupyter Notebook


This example shows how to use **finchGE** interactively to solve a simple string matching problem with **Grammatical Evolution**.

The task is to evolve a string that exactly matches a target string by defining:

* a grammar that generates candidate strings,
* a fitness function that measures similarity to the target,
* and a genetic algorithm that guides evolution.

While the problem is intentionally simple, it demonstrates the Grammatical Evolution workflow and how finchGE components can be composed programmatically.


**Problem Description**

Evolve a string to exactly match a target string by minimizing a fitness score based on character and length differences. This classic Grammatical Evolution example highlights grammar-based search, explicit genotype–phenotype mapping, and simple, interpretable fitness.


**0. Random Seed**

In finchGE, same RNG is shared across modules to ensure reproducibility.

In [ ]:
RANDOM_SEED = 42

**1. Fitness Function**

The fitness function counts character mismatches and length differences; lower values are better, with zero indicating a perfect match.


In [ ]:
from finchge.fitness import StringMatchFitness

stringmatch_fitness  = StringMatchFitness(target="finch")


**2. Grammar Definition**

The grammar defines the space of valid strings that evolution can explore.

In [ ]:
from finchge.grammar import Grammar

grammar_str = """
<string> ::= <letter> | <letter> <string>
<letter> ::= _ | [a-z]
"""

grammar = Grammar(grammar_str)

The defined grammar can be displayed using ```describe``` function.

In [ ]:
grammar.describe()

**3. Configuring Genetic Operators**

finchGE exposes genetic operators as independent components. Here we explicitly configure selection, crossover, mutation, and replacement.

In [ ]:
from finchge.operators.selection import RouletteWheelSelection
from finchge.operators.crossover import OnePointCrossover
from finchge.operators.mutation import IntFlipMutation
from finchge.operators.replacement import GenerationalReplacement



selection = RouletteWheelSelection(max_best=False)
crossover = OnePointCrossover(codon_size=127, crossover_proba=0.5)
mutation = IntFlipMutation(mutation_probability=0.01, codon_size=127)
replacement = GenerationalReplacement(max_best=False)

**4. Setting up Fitness Evaluator**

Fitness Evaluator handles the fitness evaluation

In [ ]:
from finchge.grammar.mapper import GenotypeMapper
from finchge.fitness import FitnessEvaluator

# Initialize mapper for fitness evaluator
mapper = GenotypeMapper(grammar=grammar,
                            max_wraps=6,
                            max_recursion_depth=20)
# Initialize Fitness Evaluator
fitness_fn = StringMatchFitness("hello")
fitness_evaluator = FitnessEvaluator(
    fitness_functions=fitness_fn,
    mapper=mapper
)

**5. Constructing the Genetic Algorithm**

The genetic algorithm is assembled by composing the operators.

In [ ]:
from finchge.algorithm import GeneticAlgorithm

ga = GeneticAlgorithm(
    selection=selection,
    crossover=crossover,
    mutation=mutation,
    replacement=replacement,
    elite_size=3,
    fitness_evaluator=fitness_evaluator,
    random_state=RANDOM_SEED
)

**6. Population Initialisation and Evaluation**

We now create an initial population and evaluate it.

In [ ]:
from finchge.initialisation import RandomGenomeInitialiser
from finchge.core.population import Population


initializer = RandomGenomeInitialiser(genome_length=100, codon_size=127)
population = Population(initialiser=initializer, population_size=100)


**7. Running Evolution Interactively**

Evolve the population generation by generation

In [ ]:
from tqdm import tqdm
fitness_history = [] 
fitness_evaluator.evaluate_population(population)

for generation in tqdm(range(100)):
    population = ga.evolve_one_generation(population)
    best = ga.get_best_individual(population)
    fitness_history.append(best)

**7. Inspecting the Best Individual**

After evolution completes, we can inspect the best individual found in the final generation.

In [ ]:
best_individual = fitness_history[-1]
best_individual